# CSCI 447/547 Hackathon 5: Neural Networks

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the workbook together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will implement a neural net and train it on a Titanic data set to predict whether an individual survived.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/hackathon5-neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you will need to download [ `train_clean.csv` and `test_clean.csv` from GitHub](https://github.com/lucywowen/csci547_ML/tree/main/examples/data/titanic) and upload them to Colab for this to work.

---

### Local (VSCode/Jupyter Lab) Version

To get the latest version of this notebook **without overwriting your work on other hackathons**, open a terminal in your cloned `csci547_ML` folder and run:

```bash
git fetch
git checkout origin/main -- examples/hackathon5-neural_net.ipynb
```

`git fetch` downloads the latest version of the repository without changing any of your files, then `git checkout` replaces only this notebook. *(This overwrites any changes you have made to this notebook, so if you have already started it, make a copy first!)*

Make sure that your `uv` virtual environment is enabled.

This hackathon also uses `tensorflow` (which includes `keras`) to build neural networks. If you don't have it in your environment yet, add it with:

```bash
uv add tensorflow
```

---

## 1. Neural Networks and the Titanic

On April 15, 1912, the RMS Titanic sank after hitting an iceberg, and more than 1,500 of the roughly 2,200 people on board died. Survival was not random: who you were, where you slept, and who you were travelling with all mattered.

**The Scenario:** Using passenger records, we will train a **neural network** to predict whether a passenger survived (`Survived` = 1) or died (`Survived` = 0). This is a binary classification problem, just like the logistic regression problems we've already seen.

### 1.1 Load the Libraries

We will use `keras` to build and train our neural networks. *(When you import `keras`, you may see some red log messages from TensorFlow about CUDA, GPUs, or oneDNN. These are harmless: they just mean TensorFlow will train on your CPU, which is plenty fast for this dataset.)*

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.set_theme(style='whitegrid')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

import keras
from keras import layers

keras.utils.set_random_seed(42) # makes the random parts of python, numpy, and tensorflow reproducible

### 1.2 Load the Data

The data comes in two files: `train_clean.csv` has 891 passengers **with** their `Survived` outcome, and `test_clean.csv` has 418 passengers **without** one (it comes from a Kaggle competition, so the answers are hidden).

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()

    train = pd.read_csv('train_clean.csv') # colab file import
    test = pd.read_csv('test_clean.csv')

except:

    train = pd.read_csv('data/titanic/train_clean.csv') # local file import
    test = pd.read_csv('data/titanic/test_clean.csv')

train.head()

| Feature | Description |
| :--- | :--- |
| **Age**, **Fare** | Age in years, and ticket price. |
| **Pclass** | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd). |
| **Sex** | male or female. |
| **SibSp**, **Parch** | Number of siblings/spouses, and of parents/children, aboard. |
| **Family_Size** | `SibSp + Parch`. |
| **Embarked** | Port the passenger boarded at (C = Cherbourg, Q = Queenstown, S = Southampton). |
| **Title** | Title taken from the passenger's name (Mr, Mrs, Miss, Master, Dr, Rev). |
| **Name**, **Ticket**, **Cabin**, **PassengerId** | Name, ticket number, cabin number (missing for most passengers), and a unique ID number. |
| **Survived** | 0 = died, 1 = survived. *(Target Variable)* |

### Think-Pair-Share 1: Who Survived?
**Context:** We have 13 possible input features for predicting `Survived`.

1. **Think:** Which features do you expect to be the most useful for predicting survival, and why? Are there any features that a model should **not** be given? (Look closely at `PassengerId`, `Name`, `Ticket`, and `Cabin`.)

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss your choices with your group.

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

Let's check your hypotheses:

In [ ]:
print(f"Overall survival rate: {train['Survived'].mean():.3f}")

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))
for ax, var in zip(axs, ['Pclass', 'Sex', 'Embarked']):
    sns.countplot(x=var, hue='Survived', data=train, palette='hls', ax=ax)
    ax.set_title(f'Survival by {var}')
plt.show()

## 2. Preparing the Data

Neural networks only understand numbers, and they train best when those numbers are on similar scales.

### 2.1 Encode Categorical Variables

`Sex` can become a single 0/1 column, but `Embarked` and `Title` need **one-hot encoding**: one 0/1 column per category. (Numbering the ports 0, 1, and 2 would imply that port 2 is somehow "more" than port 0.) We leave `Pclass` as a single column, since 1st > 2nd > 3rd class really is an ordering. We encode both files together so that they end up with exactly the same columns.

In [ ]:
# Stack both files into a single dataframe (test passengers get Survived = NaN)
df = pd.concat([train, test], axis=0, ignore_index=True)

# convert Sex to category codes (female = 0, male = 1)
df['Sex'] = df['Sex'].astype('category').cat.codes

# one-hot encode the categorical variables
for var in ['Embarked', 'Title']:
    df = pd.concat([df, pd.get_dummies(df[var], prefix=var, dtype=int)], axis=1)
    del df[var]

# drop the variables we won't be using
df = df.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'])

df.head()

### 2.2 Coding Exercise: Split the Data

The passengers from `test_clean.csv` have no `Survived` value, so we can't use them to check our model. Like in Hackathon 0, we'll split the 891 labeled passengers three ways: **training** (60%, to fit the network), **fine-tuning** (20%, to compare models and choose hyperparameters), and **generalization** (20%, touched only once, at the very end).

**What you need to write:** Split `X_temp` and `y_temp` into a training set (`X_train`, `y_train`) and a fine-tuning set (`X_fine`, `y_fine`) so that they end up as 60% and 20% of the *full* dataset. Use `train_test_split()` the same way as the line above it: keep the survival rate the same in both sets (`stratify`), and use `random_state=42` so your numbers match ours.

In [ ]:
# Rows with a known Survived value came from train_clean.csv; rows without one came from test_clean.csv
labeled = df[df['Survived'].notna()]
unknown = df[df['Survived'].isna()]

# Separate features (X) and target labels (y). Keras works with float32 numbers.
X = labeled.drop(columns='Survived').astype('float32')
y = labeled['Survived'].astype(int)
X_unknown = unknown.drop(columns='Survived').astype('float32')

# Hold out 20% of the labeled passengers for the final generalization test
X_temp, X_gen, y_temp, y_gen = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Split the remaining 80% into training (60% of the total) and fine-tuning (20% of the total).
# Hint: what fraction of X_temp is 20% of the full dataset?
X_train, X_fine, y_train, y_fine = # --- YOUR CODE HERE ---

print('Training set:      ', X_train.shape)
print('Fine-tuning set:   ', X_fine.shape)
print('Generalization set:', X_gen.shape)
print('Expected shapes: (534, 16), (178, 16), (179, 16)')

### 2.3 Scale Continuous Variables

Like gradient descent in the Linear Regression hackathon, neural networks train poorly when features have very different ranges (`Fare` goes up to 512, while `Parch` is usually 0 to 2), so we standardize the continuous features. The scaler learns the mean and standard deviation from the **training set only**: using the other sets would leak information about them into our model.

In [ ]:
continuous = ['Age', 'Fare', 'Parch', 'Pclass', 'SibSp', 'Family_Size']

# make independent copies so we can safely overwrite columns
X_train, X_fine, X_gen, X_unknown = X_train.copy(), X_fine.copy(), X_gen.copy(), X_unknown.copy()

scaler = StandardScaler()
X_train[continuous] = scaler.fit_transform(X_train[continuous]) # learn the mean and std from the training set...
X_fine[continuous] = scaler.transform(X_fine[continuous])       # ...and apply them to everything else
X_gen[continuous] = scaler.transform(X_gen[continuous])
X_unknown[continuous] = scaler.transform(X_unknown[continuous])

## 3. What Is a Neural Network?

Recall logistic regression: take a weighted sum of the inputs, then squash it into a probability with the sigmoid function. A single **neuron** does exactly that:

$$a = g(\mathbf{w}^T\mathbf{x} + b)$$

where $\mathbf{w}$ are the neuron's **weights**, $b$ is its **bias**, and $g$ is its **activation function**. A **layer** is a group of neurons that all look at the same inputs, each with its own weights. A **neural network** stacks layers: the outputs of one layer become the inputs to the next. The layers in the middle are called **hidden layers**, and the last layer (here, a single neuron) outputs the probability of survival.

That output neuron is just logistic regression, but instead of using our raw features, it uses features *learned* by the hidden layers. (In Hackathon 3, we hand-crafted polynomial features. A neural network learns its own.) The two most common activation functions are:

- **Sigmoid**, which squashes any number into (0, 1). We use it on the output neuron to get a probability.
- **ReLU** (Rectified Linear Unit), which outputs 0 for negative inputs and leaves positive inputs unchanged. It is the most common choice for hidden layers.

### Think-Pair-Share 2: Why Do We Need Activation Functions?
**Context:** Suppose the hidden neurons used a *linear* activation, meaning no activation at all: each hidden neuron just outputs its weighted sum $\mathbf{w}^T\mathbf{x} + b$.

1. **Think:** The output neuron then takes a weighted sum of those weighted sums. What kind of function of the original features is that? What model would the whole network be equivalent to? Would adding more linear hidden layers help?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss with your group. What does the "kink" in ReLU at 0 let a network do that a stack of linear layers can't?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

## 4. Building and Training a Network with Keras

### 4.1 Create the Model

The function below (handled for you) builds a network with whichever hyperparameters we choose, which will come in handy when we tune them later. In Keras, a `Sequential` model is a stack of layers, and each `layers.Dense()` is a layer of neurons connected to every input. By default, it builds 16 inputs (one per feature) $\to$ 8 hidden ReLU neurons $\to$ 1 sigmoid output neuron. `model.compile()` chooses the **loss function** (binary cross-entropy, which is the log loss from logistic regression), the **optimizer** (Adam, a version of gradient descent that adapts its learning rate automatically), and the metric to report.

In [ ]:
def create_model(lyrs=[8], act='relu', opt='adam', dr=0.0):
    """
    Builds and compiles a fully-connected neural network for binary classification.
    lyrs: number of neurons in each hidden layer, e.g. [32, 16] means two hidden layers
    act: activation function for the hidden layers
    opt: optimization algorithm used during training
    dr: dropout rate (the fraction of neurons randomly switched off during training)
    """
    keras.utils.set_random_seed(42) # for reproducibility

    model = keras.Sequential()
    model.add(keras.Input(shape=(X_train.shape[1],))) # one input per feature

    for n_neurons in lyrs:
        model.add(layers.Dense(n_neurons, activation=act)) # hidden layers

    model.add(layers.Dropout(dr)) # dropout (more on this in Section 5), default is none
    model.add(layers.Dense(1, activation='sigmoid')) # output layer: the probability of survival

    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    return model


model = create_model()
model.summary()

### 4.2 Train the Model

Training a network is gradient descent, repeated over many **epochs** (passes through the training data). Each gradient descent step uses a **batch** of passengers (here, 32), and a technique called **backpropagation** works out how to adjust every weight in every layer. After each epoch, Keras also reports the loss and accuracy on the fine-tuning set (`validation_data`), which is only used for reporting, never for updating the weights.

Here's how we train the model with Keras's `fit()` method (handled for you). After training, a model's `evaluate()` method gives its loss and accuracy on any dataset.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_fine, y_fine), # report the loss and accuracy on the fine-tuning set after every epoch
    epochs=100,                       # 100 passes through the training data
    batch_size=32,                    # 32 passengers per gradient descent step
    verbose=0,                        # don't print 100 progress bars
)

# evaluate() returns the [loss, accuracy] of the trained model on any dataset
fine_loss, fine_acc = model.evaluate(X_fine, y_fine, verbose=0)

print(f"Final training accuracy: {history.history['accuracy'][-1]:.3f}")
print(f'Fine-tuning accuracy:    {fine_acc:.3f}')

### 4.3 Learning Curves

`fit()` returns the training **history**: the loss and accuracy on the training and fine-tuning sets after every epoch. Plotting these **learning curves** is the best way to see how training went: if both curves are still improving at the end, the network could train for longer, and if the fine-tuning curve gets *worse* while the training curve keeps improving, the network has started to overfit.

In [ ]:
def plot_history(history, title):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
    for ax, metric in zip(axs, ['loss', 'accuracy']):
        ax.plot(history.history[metric], label='train')
        ax.plot(history.history['val_' + metric], label='fine-tune (validation)')
        ax.set(xlabel='epoch', ylabel=metric, title=f'{title}: {metric}')
        ax.legend()
    plt.show()

plot_history(history, 'Small network')

### 4.4 Coding Exercise: What Overfitting Looks Like

Our small network has 145 weights and biases to fit 534 training passengers. What happens with a much bigger network, trained for longer?

**What you need to write:** Use `create_model()` to build `big_model`, a network with **two hidden layers of 128 neurons each** (18,817 weights and biases!), and train it the same way as the model in Section 4.2, but for 200 epochs. Keep its training history in `big_history` so the plot below can use it.

In [ ]:
# --- YOUR CODE HERE ---

plot_history(big_history, 'Big network')

### Think-Pair-Share 3: Reading Learning Curves
**Context:** Compare the learning curves of the big network to those of the small network in Section 4.3.

1. **Think:** Look at the big network's fine-tuning (validation) **loss**. At roughly what epoch is it lowest, and what happens after that? Meanwhile, what is the training loss doing? What is the network doing to the training data?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss with your group. List at least two changes that could fix this problem. (Hint: look at the arguments of `create_model()` and `fit()`.)

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

## 5. Improving the Model: Hyperparameter Tuning

The number of layers, the number of neurons, the activation function, and the number of epochs are all **hyperparameters**: settings that we choose, rather than weights the network learns. Training networks is slow, so instead of a full grid search, we'll try a handful of configurations, training each one on the **training set** and scoring it on the **fine-tuning set**. Following step 6 of our workflow, we will **not** look at the generalization set while making these decisions.

One hyperparameter we'll try is **dropout**, which randomly switches off a fraction `dr` of the neurons during each training step, so that the network can't rely on any single neuron to memorize the training data. Like ridge and lasso, dropout is a form of **regularization**. This cell takes about 30 seconds.

In [ ]:
configs = [
    {'lyrs': [8],      'act': 'linear', 'dr': 0.0},
    {'lyrs': [8],      'act': 'relu',   'dr': 0.0},
    {'lyrs': [32, 16], 'act': 'relu',   'dr': 0.0},
    {'lyrs': [32, 16], 'act': 'relu',   'dr': 0.3},
    {'lyrs': [64, 64], 'act': 'relu',   'dr': 0.5},
]

results = []
for cfg in configs:
    model = create_model(lyrs=cfg['lyrs'], act=cfg['act'], dr=cfg['dr'])
    model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0)
    fine_loss, fine_acc = model.evaluate(X_fine, y_fine, verbose=0)
    results.append({**cfg, 'fine_acc': fine_acc}) # copy the settings and add the score

results_df = pd.DataFrame(results).sort_values('fine_acc', ascending=False)
results_df

**Pause point:** The fine-tuning set has 178 passengers, so each passenger is worth about 0.56% accuracy. How many passengers separate the best configuration from the worst? Does the `linear` network do worse than the `relu` ones, as Think-Pair-Share 2 predicted?

### 5.1 Train the Final Model

Now we retrain the best configuration on the training **and** fine-tuning data combined, just like in Hackathon 0.

In [ ]:
best = results_df.iloc[0]

X_combined = pd.concat([X_train, X_fine])
y_combined = pd.concat([y_train, y_fine])

final_model = create_model(lyrs=best['lyrs'], act=best['act'], dr=best['dr'])
final_model.fit(X_combined, y_combined, epochs=100, batch_size=32, verbose=0)

print('Final model trained with the best configuration:', best['lyrs'], best['act'], best['dr'])

## 6. Evaluate Generalization

Now, and only now, we use the generalization set. The network outputs a survival **probability** for each passenger, so we predict "survived" whenever that probability is above 0.5.

In [ ]:
gen_probs = final_model.predict(X_gen, verbose=0)
gen_pred = (gen_probs > 0.5).astype(int).ravel() # True/False -> 1/0, flattened to 1D

gen_acc = accuracy_score(y_gen, gen_pred)
print(f'Generalization accuracy: {gen_acc:.3f}')

ConfusionMatrixDisplay.from_predictions(y_gen, gen_pred, display_labels=['Died', 'Survived'])
plt.title('Neural Network Confusion Matrix (Generalization Set)')
plt.grid(False)
plt.show()

### Think-Pair-Share 4: Is That Good?
**Context:** Our network got around 80% of the generalization set right.

1. **Think:** What accuracy would a "model" get if it simply predicted that *every* passenger died? (Hint: look at the survival rate from Section 1.2.) Is 80% impressive compared to that?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Would you expect plain logistic regression to do better or worse than our neural network on this data? Agree on a prediction as a group, then run the cell below to check it.

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

In [ ]:
baseline_acc = (y_gen == 0).mean()
print(f"'Everyone died' baseline accuracy:            {baseline_acc:.3f}")

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_combined, y_combined)
print(f'Logistic regression generalization accuracy: {log_reg.score(X_gen, y_gen):.3f}')
print(f'Neural network generalization accuracy:      {gen_acc:.3f}')

### 6.1 Predict the Unknown Passengers

Finally, the 418 passengers from `test_clean.csv`. We can't score these predictions since their outcomes are hidden, but this is what a model is ultimately for: making predictions about new data.

In [ ]:
unknown_pred = (final_model.predict(X_unknown, verbose=0) > 0.5).astype(int).ravel()
print(f'Predicted survivors: {unknown_pred.sum()} of {len(unknown_pred)} passengers')

test[['Name', 'Sex', 'Age', 'Pclass']].assign(Predicted_Survived=unknown_pred).head(10)

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions.

1. **Trade-off Analysis:**
In Think-Pair-Share 4, plain logistic regression did as well as (or better than) our neural network. A neural network has more weights, more hyperparameters to tune, and takes longer to train. Why might all that extra flexibility not help on a dataset of ~900 passengers with 16 features? When *is* it worth it?

    **ANSWER**

2. **Diagnosing Failures:**
A classmate trains a network and reports 97% training accuracy but only 76% fine-tuning accuracy, and their fine-tuning loss has risen every epoch since epoch 15. What is the problem? Name two specific changes to `create_model()` or `fit()` that could fix it.

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world problem where you would choose a neural network over logistic regression:

    1. 